In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install gradio praat-parselmouth librosa shap -q
print("✅ 完成")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 98.1 MB/s eta 0:00:00
✅ 完成


In [ ]:
import pickle
import numpy as np
import librosa
import parselmouth
from parselmouth.praat import call
import shap

BASE_DIR = '/content/drive/MyDrive/voice_age_project_0428/voice_age_project_0428'

with open(f'{BASE_DIR}/model_artifacts_binary.pkl', 'rb') as f:
    artifacts = pickle.load(f)

xgb_model    = artifacts['xgb_model']
scaler       = artifacts['scaler']
le           = artifacts['label_encoder']
feature_cols = artifacts['feature_cols']

print("✅ 模型載入完成")
print(f"類別：{le.classes_}")
print(f"特徵數量：{len(feature_cols)} 維")
print(f"特徵列表：{feature_cols}")

✅ 模型載入完成
類別：['51 and Above' 'Below 51']
特徵數量：16 維
特徵列表：['F0_mean', 'Jitter', 'Shimmer', 'HNR', 'MFCC_2', 'MFCC_3', 'MFCC_4', 'MFCC_5', 'MFCC_6', 'MFCC_7', 'MFCC_8', 'MFCC_9', 'MFCC_10', 'MFCC_11', 'MFCC_12', 'MFCC_13']


In [ ]:
def extract_features(wav_path):
    try:
        sound = parselmouth.Sound(wav_path)

        pitch   = call(sound, "To Pitch", 0.0, 75, 600)
        f0_mean = call(pitch, "Get mean", 0, 0, "Hertz")

        point_process = call(sound, "To PointProcess (periodic, cc)", 75, 600)
        jitter  = call(point_process, "Get jitter (local)",
                       0, 0, 0.0001, 0.02, 1.3)
        shimmer = call([sound, point_process], "Get shimmer (local)",
                       0, 0, 0.0001, 0.02, 1.3, 1.6)

        harmonicity = call(sound, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0)
        hnr         = call(harmonicity, "Get mean", 0, 0)

        y, sr      = librosa.load(wav_path, sr=16000)
        mfccs      = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfcc_means = np.mean(mfccs, axis=1)

        # MFCC_1（index 0）已移除，從 MFCC_2（index 1）開始
        features = {
            'F0_mean': f0_mean if not np.isnan(f0_mean) else 0.0,
            'Jitter':  jitter  if not np.isnan(jitter)  else 0.0,
            'Shimmer': shimmer if not np.isnan(shimmer) else 0.0,
            'HNR':     hnr     if not np.isnan(hnr)     else 0.0,
        }
        for i in range(1, 13):   # i=1 → MFCC_2, i=12 → MFCC_13
            features[f'MFCC_{i+1}'] = mfcc_means[i]

        return features
    except:
        return None

In [ ]:
def predict_age(wav_path):
    feats = extract_features(wav_path)
    if feats is None:
        return None, None, None, None, None

    X        = np.array([[feats[col] for col in feature_cols]])
    X_scaled = scaler.transform(X)

    pred_idx   = xgb_model.predict(X_scaled)[0]
    pred_proba = xgb_model.predict_proba(X_scaled)[0]
    pred_label = le.inverse_transform([pred_idx])[0]

    explainer = shap.TreeExplainer(xgb_model)
    shap_vals = explainer.shap_values(X_scaled)

    return pred_label, pred_proba, shap_vals, X_scaled, feats

In [ ]:
# 用一段已知音檔測試，確認特徵維度正確
import os

test_wav = None
audio_dir = f'{BASE_DIR}/AudioWAV'
for f in os.listdir(audio_dir)[:1]:
    if f.endswith('.wav'):
        test_wav = os.path.join(audio_dir, f)

if test_wav:
    result = predict_age(test_wav)
    if result[0]:
        pred_label, pred_proba, shap_vals, _, feats = result
        print(f"測試預測：{pred_label}")
        print(f"機率：{dict(zip(le.classes_, pred_proba.round(3)))}")
        print(f"shap_vals shape：{shap_vals.shape}")
        print(f"特徵值（前4個）：F0={feats['F0_mean']:.1f} Jitter={feats['Jitter']*100:.3f}% HNR={feats['HNR']:.1f}dB")
    else:
        print("❌ 特徵提取失敗")

測試預測：Below 51
機率：{'51 and Above': np.float32(0.012), 'Below 51': np.float32(0.988)}
shap_vals shape：(1, 16)
特徵值（前4個）：F0=228.5 Jitter=3.192% HNR=9.7dB


In [ ]:
import gradio as gr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
matplotlib.use('Agg')

C_BLUE   = '#445C9C'
C_ORANGE = '#D98C6B'
C_SAGE   = '#8FA99B'
C_MIST   = '#A8B7C7'
C_DARK   = '#2A2A2A'

GROUP_COLOR = {
    'Below 51':     C_SAGE,
    '51 and Above': C_ORANGE
}

TEXT = {
    '中文': {
        'title':       '# AI 聲帶年齡分析儀',
        'subtitle':    '### 聲紋生物標記與聲帶生理年齡檢測系統',
        'desc':        '對著麥克風說話約 3 秒，系統分析 16 個聲學特徵，判斷聲帶生理狀態屬於 51 歲以下或 51 歲以上，並說明 AI 的判斷依據。',
        'audio_label': '錄音或上傳音檔（說話約 3 秒）',
        'btn':         '開始分析',
        'result_label':'預測結果',
        'interp_label':'數據解釋',
        'prob_label':  '預測機率圖',
        'shap_label':  'AI 判斷依據（SHAP）',
        'no_audio':    '請先錄音或上傳音檔',
        'fail':        '音檔分析失敗，請重試',
        'group': {'Below 51': '51 歲以下', '51 and Above': '51 歲以上'},
        'result_template': lambda lz, bp, ap, f0, j, sh, hn: f"""預測結果：{lz}

51 歲以下機率：{bp:.1f}%
51 歲以上機率：{ap:.1f}%

聲學生物標記數值：
  F0 基頻           {f0:.1f} Hz
  Jitter 頻率微擾   {j:.3f} %
  Shimmer 振幅微擾  {sh:.3f} %
  HNR 諧波雜訊比    {hn:.1f} dB""",
        'pie_title':   'Vocal Age Prediction Probability',
        'shap_title':  'SHAP Feature Contribution',
        'shap_xlabel': 'SHAP value  (positive → 51+,  negative → Below 51)',
        'legend_above':'Pushes toward 51+',
        'legend_below':'Pushes toward Below 51',
        'interp_header_bio':  '聲學指標解讀：',
        'interp_header_shap': '\nAI 判斷依據解讀：',
        'shap_top3':   '影響最大的三個特徵：',
        'shap_rank':   lambda r, f, d, v: f"  第 {r} 名：{f}（{d}，影響強度 {v:.3f}）",
        'dir_above':   '推向 51 歲以上',
        'dir_below':   '推向 51 歲以下',
        'shap_legend': ['橘色長條：該特徵將預測推向 51 歲以上。',
                        '藍灰色長條：該特徵將預測推向 51 歲以下。',
                        '長條越長，影響越大。'],
        'f0_low':   lambda v: f"F0 基頻 {v:.1f} Hz：偏低，可能為女性更年期後聲帶水腫的老化特徵。",
        'f0_high':  lambda v: f"F0 基頻 {v:.1f} Hz：偏高，可能為男性聲帶萎縮後的老化特徵。",
        'f0_ok':    lambda v: f"F0 基頻 {v:.1f} Hz：在正常範圍內（男性 85–180 Hz，女性 165–255 Hz）。",
        'j_high':   lambda v: f"Jitter {v:.3f}%：超過臨床異常值（1.04%），顯示神經控制精度下降。",
        'j_ok':     lambda v: f"Jitter {v:.3f}%：在正常範圍內（< 1.04%），聲帶振動頻率穩定。",
        'sh_high':  lambda v: f"Shimmer {v:.3f}%：超過正常值（3.81%），顯示聲帶肌力不穩定。",
        'sh_ok':    lambda v: f"Shimmer {v:.3f}%：在正常範圍內（< 3.81%），聲帶肌力穩定。",
        'hnr_low':  lambda v: f"HNR {v:.1f} dB：明顯偏低，聲帶閉合不全，漏氣產生雜訊，為老化核心症狀。",
        'hnr_mid':  lambda v: f"HNR {v:.1f} dB：略低於正常值（20 dB），聲帶閉合效率稍差。",
        'hnr_ok':   lambda v: f"HNR {v:.1f} dB：正常（> 20 dB），聲帶閉合良好。",
    },
    'English': {
        'title':       '# AI Vocal Age Analyzer',
        'subtitle':    '### Vocal Biomarker & Age Detection System',
        'desc':        'Speak into the microphone for ~3 seconds. The system analyzes 16 acoustic features to predict whether your vocal cords are Below 51 or 51 and Above.',
        'audio_label': 'Record or upload audio (~3 seconds)',
        'btn':         'Analyze',
        'result_label':'Prediction Result',
        'interp_label':'Data Interpretation',
        'prob_label':  'Prediction Probability',
        'shap_label':  'AI Decision Basis (SHAP)',
        'no_audio':    'Please record or upload audio first.',
        'fail':        'Analysis failed. Please try again.',
        'group': {'Below 51': 'Below 51', '51 and Above': '51 and Above'},
        'result_template': lambda lz, bp, ap, f0, j, sh, hn: f"""Prediction: {lz}

Below 51 probability:     {bp:.1f}%
51 and Above probability: {ap:.1f}%

Vocal Biomarker Values:
  F0 mean pitch:    {f0:.1f} Hz
  Jitter:           {j:.3f} %
  Shimmer:          {sh:.3f} %
  HNR:              {hn:.1f} dB""",
        'pie_title':   'Vocal Age Prediction Probability',
        'shap_title':  'SHAP Feature Contribution',
        'shap_xlabel': 'SHAP value  (positive → 51+,  negative → Below 51)',
        'legend_above':'Pushes toward 51+',
        'legend_below':'Pushes toward Below 51',
        'interp_header_bio':  'Biomarker Interpretation:',
        'interp_header_shap': '\nAI Decision Basis:',
        'shap_top3':   'Top 3 most influential features:',
        'shap_rank':   lambda r, f, d, v: f"  #{r}: {f} ({d}, impact {v:.3f})",
        'dir_above':   'pushes toward 51+',
        'dir_below':   'pushes toward Below 51',
        'shap_legend': ['Orange bars: pushes prediction toward 51+.',
                        'Blue-grey bars: pushes prediction toward Below 51.',
                        'Longer bar = stronger influence.'],
        'f0_low':   lambda v: f"F0 {v:.1f} Hz: Below normal. May indicate post-menopausal vocal fold edema.",
        'f0_high':  lambda v: f"F0 {v:.1f} Hz: Above normal. May indicate vocal fold atrophy in aging males.",
        'f0_ok':    lambda v: f"F0 {v:.1f} Hz: Within normal range (male 85–180 Hz, female 165–255 Hz).",
        'j_high':   lambda v: f"Jitter {v:.3f}%: Above threshold (1.04%). Suggests neuromotor decline.",
        'j_ok':     lambda v: f"Jitter {v:.3f}%: Normal (< 1.04%). Stable vocal fold vibration.",
        'sh_high':  lambda v: f"Shimmer {v:.3f}%: Above normal (3.81%). Reduced vocal fold muscle strength.",
        'sh_ok':    lambda v: f"Shimmer {v:.3f}%: Normal (< 3.81%). Stable vocal fold strength.",
        'hnr_low':  lambda v: f"HNR {v:.1f} dB: Significantly low. Incomplete glottal closure — core aging symptom.",
        'hnr_mid':  lambda v: f"HNR {v:.1f} dB: Slightly below normal (20 dB). Reduced glottal closure.",
        'hnr_ok':   lambda v: f"HNR {v:.1f} dB: Normal (> 20 dB). Good glottal closure.",
    }
}

SCIENCE_MD = {
    '中文': """
---
## 聲帶老化科普知識

**什麼是老年性嗓音（Presbyphonia）？**

聲帶隨年齡退化產生的嗓音功能下降，是正常的生理老化過程，不是疾病。
典型症狀包括聲音沙啞、音量減弱、說話費力。

---

**四個關鍵聲學指標**

| 指標 | 全名 | 老化方向 | 意義 |
|------|------|----------|------|
| F0 | 基頻（音高） | 男性升高，女性降低 | 聲帶每秒振動的次數 |
| Jitter | 頻率微擾 | 升高 | 每次振動週期的不穩定程度 |
| Shimmer | 振幅微擾 | 升高 | 每次振動音量的不穩定程度 |
| HNR | 諧波雜訊比 | 降低 | 聲音中純音與雜訊的比例 |

---

**什麼是 MFCC？**

梅爾頻率倒譜係數（Mel-Frequency Cepstral Coefficients）。
把聲音的頻率分佈壓縮成數字，捕捉聲道共鳴特性。
本系統使用 MFCC_2 至 MFCC_13 共 12 維。

---

**什麼是 SHAP？**

SHapley Additive exPlanations，基於賽局理論的可解釋性 AI 方法。
它計算每個特徵對預測結果的貢獻大小，讓 AI 的判斷邏輯透明可驗證。

---

**為什麼以 51 歲為分界？**

51 歲是聲帶老化特徵開始顯著出現的臨床分水嶺。
模型準確率達 85% 以上，臨床意義明確：
判斷聲帶是否已進入高齡退化階段。
""",
    'English': """
---
## Science Behind Vocal Aging

**What is Presbyphonia?**

Age-related decline in vocal function. A normal physiological aging process, not a disease.
Typical symptoms: hoarseness, reduced volume, vocal fatigue.

---

**Four Key Acoustic Features**

| Feature | Full Name | Aging Direction | Meaning |
|---------|-----------|----------------|---------|
| F0 | Fundamental frequency | ↑ males, ↓ females | Vocal fold vibration rate |
| Jitter | Frequency perturbation | ↑ | Cycle-to-cycle frequency instability |
| Shimmer | Amplitude perturbation | ↑ | Cycle-to-cycle amplitude instability |
| HNR | Harmonics-to-noise ratio | ↓ | Ratio of pure tone to noise |

---

**What is MFCC?**

Mel-Frequency Cepstral Coefficients.
Captures vocal tract resonance characteristics.
This system uses MFCC_2 through MFCC_13 (12 dimensions).

---

**What is SHAP?**

SHapley Additive exPlanations — explainable AI based on game theory.
Calculates each feature's contribution to the prediction,
making the AI's logic transparent and scientifically auditable.

---

**Why threshold at age 51?**

Age 51 is a clinical landmark for significant vocal aging.
Model accuracy exceeds 85%.
Clinical question: has the vocal fold entered the aging stage?
"""
}


def interpret_biomarkers(feats, lang):
    T = TEXT[lang]
    lines = []
    f0 = feats['F0_mean']
    lines.append(T['f0_low'](f0) if f0 < 85 else T['f0_high'](f0) if f0 > 255 else T['f0_ok'](f0))
    j = feats['Jitter'] * 100
    lines.append(T['j_high'](j) if j > 1.04 else T['j_ok'](j))
    sh = feats['Shimmer'] * 100
    lines.append(T['sh_high'](sh) if sh > 3.81 else T['sh_ok'](sh))
    hn = feats['HNR']
    lines.append(T['hnr_low'](hn) if hn < 15 else T['hnr_mid'](hn) if hn < 20 else T['hnr_ok'](hn))
    return "\n".join(lines)


def interpret_shap(shap_for_plot, lang):
    T = TEXT[lang]
    sorted_idx = np.argsort(np.abs(shap_for_plot))[::-1][:3]
    lines = [T['shap_top3']]
    for rank, idx in enumerate(sorted_idx, 1):
        feat = feature_cols[idx]
        val  = shap_for_plot[idx]
        direction = T['dir_above'] if val > 0 else T['dir_below']
        lines.append(T['shap_rank'](rank, feat, direction, abs(val)))
    lines.append("")
    lines.extend(T['shap_legend'])
    return "\n".join(lines)


def analyze_voice(audio_path, lang):
    T = TEXT[lang]
    if audio_path is None:
        return T['no_audio'], None, None, ""
    result = predict_age(audio_path)
    if result[0] is None:
        return T['fail'], None, None, ""

    pred_label, pred_proba, shap_vals, X_scaled, feats = result
    label_zh   = T['group'][pred_label]
    below_idx  = list(le.classes_).index('Below 51')
    above_idx  = list(le.classes_).index('51 and Above')
    below_prob = pred_proba[below_idx] * 100
    above_prob = pred_proba[above_idx] * 100

    result_text = T['result_template'](
        label_zh, below_prob, above_prob,
        feats['F0_mean'], feats['Jitter']*100,
        feats['Shimmer']*100, feats['HNR']
    )

    # 機率圓餅圖
    fig1, ax1 = plt.subplots(figsize=(5, 4), facecolor='white')
    colors = [GROUP_COLOR[c] for c in le.classes_]
    ax1.pie(pred_proba, colors=colors, autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2},
            textprops={'fontsize': 12, 'color': 'white', 'fontweight': 'bold'})
    ax1.legend(
        handles=[mpatches.Patch(color=GROUP_COLOR[c], label=c) for c in le.classes_],
        loc='lower center', bbox_to_anchor=(0.5, -0.12), fontsize=10, frameon=False
    )
    ax1.set_title(T['pie_title'], fontsize=13, color=C_BLUE, fontweight='bold', pad=15)
    plt.tight_layout()

    # SHAP 條形圖（取負使正值代表推向 51+）
    below_idx_shap = list(le.classes_).index('Below 51')
    if shap_vals.ndim == 3:
        shap_for_plot = -shap_vals[0, :, below_idx_shap]
    else:
        shap_for_plot = -shap_vals[0]

    sorted_idx = np.argsort(np.abs(shap_for_plot))[-10:]
    bar_colors = [C_ORANGE if v > 0 else C_MIST for v in shap_for_plot[sorted_idx]]

    fig2, ax2 = plt.subplots(figsize=(6, 4.5), facecolor='white')
    ax2.barh([feature_cols[i] for i in sorted_idx], shap_for_plot[sorted_idx],
             color=bar_colors, edgecolor='white', linewidth=0.8)
    ax2.axvline(0, color=C_DARK, linewidth=1.0)
    ax2.set_xlabel(T['shap_xlabel'], fontsize=9, color=C_DARK)
    ax2.set_title(T['shap_title'], fontsize=12, color=C_BLUE, fontweight='bold')
    ax2.tick_params(labelsize=9, colors=C_DARK)
    for spine in ['top', 'right']:
        ax2.spines[spine].set_visible(False)
    ax2.spines['left'].set_color('#A8B7C7')
    ax2.spines['bottom'].set_color('#A8B7C7')
    ax2.legend(
        handles=[mpatches.Patch(color=C_ORANGE, label=T['legend_above']),
                 mpatches.Patch(color='#A8B7C7', label=T['legend_below'])],
        fontsize=9, frameon=False, loc='lower right'
    )
    plt.tight_layout()

    interpretation = (
        T['interp_header_bio'] + "\n" +
        interpret_biomarkers(feats, lang) +
        T['interp_header_shap'] + "\n" +
        interpret_shap(shap_for_plot, lang)
    )

    return result_text, fig1, fig2, interpretation


def update_ui(lang):
    T = TEXT[lang]
    return (
        gr.update(value=f"{T['title']}\n{T['subtitle']}\n{T['desc']}"),
        gr.update(label=T['audio_label']),
        gr.update(value=T['btn']),
        gr.update(label=T['result_label']),
        gr.update(label=T['interp_label']),
        gr.update(label=T['prob_label']),
        gr.update(label=T['shap_label']),
        gr.update(value=SCIENCE_MD[lang]),
    )


with gr.Blocks(
    theme=gr.themes.Base(
        primary_hue=gr.themes.colors.Color(
            c50="#EEF1F8", c100="#D8DCF0", c200="#B8C2E4",
            c300="#96A7D8", c400="#738EC8", c500="#445C9C",
            c600="#374A7E", c700="#2B3A62", c800="#1F2B48",
            c900="#141D30", c950="#0A0F1A"
        ),
        neutral_hue=gr.themes.colors.Color(
            c50="#F5F3F0", c100="#E8E3DC", c200="#D8C8B6",
            c300="#C4AE95", c400="#B09578", c500="#8FA99B",
            c600="#7A9287", c700="#647B72", c800="#4E625A",
            c900="#384A44", c950="#23302C"
        ),
        font=gr.themes.GoogleFont("Noto Sans TC")
    ),
    title="AI Vocal Age Analyzer"
) as demo:

    lang_toggle = gr.Radio(
        choices=["中文", "English"], value="中文",
        label="語言 / Language", interactive=True
    )

    header_md = gr.Markdown(
        f"{TEXT['中文']['title']}\n{TEXT['中文']['subtitle']}\n{TEXT['中文']['desc']}"
    )

    with gr.Row():
        audio_input = gr.Audio(
            sources=["microphone", "upload"],
            type="filepath",
            label=TEXT['中文']['audio_label']
        )

    submit_btn = gr.Button(TEXT['中文']['btn'], variant="primary")

    with gr.Row():
        result_text    = gr.Textbox(label=TEXT['中文']['result_label'],  lines=12)
        interpretation = gr.Textbox(label=TEXT['中文']['interp_label'], lines=12)

    with gr.Row():
        prob_plot = gr.Plot(label=TEXT['中文']['prob_label'])
        shap_plot = gr.Plot(label=TEXT['中文']['shap_label'])

    science_md = gr.Markdown(SCIENCE_MD['中文'])

    lang_toggle.change(
        fn=update_ui,
        inputs=[lang_toggle],
        outputs=[header_md, audio_input, submit_btn,
                 result_text, interpretation,
                 prob_plot, shap_plot, science_md]
    )

    submit_btn.click(
        fn=analyze_voice,
        inputs=[audio_input, lang_toggle],
        outputs=[result_text, prob_plot, shap_plot, interpretation]
    )

demo.launch(share=True, debug=False)

/tmp/ipykernel_1617/3794707091.py:322: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://10015a79023c61ad33.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
